In [ ]:
import networkx as nx
import numpy as np
import pandas as pd

# ==================================================
# HELPER FUNCTIONS
# ==================================================

def largest_connected_component(G: nx.Graph) -> nx.Graph:
    """
    Return the Largest Connected Component (LCC) as a subgraph.
    This is required for path-based metrics (avg path, diameter),
    which are only defined on connected graphs.
    """
    if G.number_of_nodes() == 0:
        return G.copy()

    components = list(nx.connected_components(G))
    if not components:
        return G.copy()

    lcc_nodes = max(components, key=len)
    return G.subgraph(lcc_nodes).copy()


def top_k_nodes(metric_dict, k=5):
    """
    Return top-k nodes sorted by metric value (descending).
    Output format: [(node, value), ...]
    """
    return sorted(metric_dict.items(), key=lambda x: x[1], reverse=True)[:k]


# ==================================================
# MAIN GRAPH SUMMARY FUNCTION
# ==================================================

def summarize_graph(G: nx.Graph, phase: str = "") -> dict:
    """
    Compute key network statistics for the given graph.

    Notes:
    - Graph is treated as undirected and unweighted
    - Path-based metrics are computed on the Largest Connected Component (LCC)
    - Includes assortativity and centrality rankings

    Returns:
        Dictionary of network metrics (ready for dataframe / CSV)
    """

    # Ensure undirected representation
    Gs = nx.Graph(G)

    n = Gs.number_of_nodes()
    m = Gs.number_of_edges()

    # Degree statistics
    degree_list = [d for _, d in Gs.degree()]

    # Extract Largest Connected Component
    LCC = largest_connected_component(Gs)

    # --- Path-based metrics ---
    try:
        avg_path = (
            nx.average_shortest_path_length(LCC)
            if LCC.number_of_nodes() > 1
            else np.nan
        )
    except Exception:
        avg_path = np.nan

    diameter = (
        nx.diameter(LCC)
        if nx.is_connected(LCC) and LCC.number_of_nodes() > 1
        else np.nan
    )

    # --- Global structure ---
    density = nx.density(Gs)
    clustering = nx.average_clustering(Gs)

    # --- Centrality measures ---
    # Use sampling for large graphs to reduce computation time
    if n > 2500:
        k_sample = min(1000, n)
        betweenness = nx.betweenness_centrality(Gs, k=k_sample, seed=42)
    else:
        betweenness = nx.betweenness_centrality(Gs)

    degree_centrality = nx.degree_centrality(Gs)

    top_degree = top_k_nodes(degree_centrality, k=5)
    top_betweenness = top_k_nodes(betweenness, k=5)

    # --- Assortativity ---
    # By node attribute (e.g., political side)
    try:
        assortativity_side = nx.attribute_assortativity_coefficient(Gs, "side")
    except Exception:
        assortativity_side = np.nan

    # By node degree
    try:
        assortativity_degree = nx.degree_assortativity_coefficient(Gs)
    except Exception:
        assortativity_degree = np.nan

    # --- Final output ---
    return {
        "phase": phase,
        "model": "Real",
        "nodes": n,
        "edges": m,
        "avg_degree": float(np.mean(degree_list)) if degree_list else 0.0,
        "density": density,
        "clustering": clustering,
        "avg_path": avg_path,
        "diameter": diameter,
        "assortativity_side": assortativity_side,
        "assortativity_degree": assortativity_degree,
        "top5_degree": "; ".join([f"{u}:{v:.4f}" for u, v in top_degree]),
        "top5_betweenness": "; ".join([f"{u}:{v:.6f}" for u, v in top_betweenness]),
    }


# ==================================================
# MODEL COMPARISON (ER vs BA)
# ==================================================

def compare_models(n: int, m: int, phase: str):
    """
    Generate synthetic graphs with similar size:
    - Erdős–Rényi (ER)
    - Barabási–Albert (BA)

    Used to compare real network structure vs theoretical models.
    """

    if n < 2:
        return []

    # ER probability (expected density match)
    p = 2 * m / (n * (n - 1)) if n > 1 else 0.0

    # BA parameter (edges added per new node)
    m_ba = max(1, round(m / n)) if n else 1

    # Generate models
    G_er = nx.erdos_renyi_graph(n, p, seed=42)
    G_ba = nx.barabasi_albert_graph(n, m_ba, seed=42)

    def compute_stats(G: nx.Graph, label: str) -> dict:
        """
        Compute simplified statistics for synthetic graphs.
        Note: no 'side' attribute available.
        """
        Gs = nx.Graph(G)
        LCC = largest_connected_component(Gs)

        try:
            avg_path = (
                nx.average_shortest_path_length(LCC)
                if LCC.number_of_nodes() > 1
                else np.nan
            )
        except Exception:
            avg_path = np.nan

        diameter = (
            nx.diameter(LCC)
            if nx.is_connected(LCC) and LCC.number_of_nodes() > 1
            else np.nan
        )

        degree_list = [d for _, d in Gs.degree()]

        return {
            "phase": phase,
            "model": label,
            "nodes": Gs.number_of_nodes(),
            "edges": Gs.number_of_edges(),
            "avg_degree": float(np.mean(degree_list)) if degree_list else 0.0,
            "density": nx.density(Gs),
            "clustering": nx.average_clustering(Gs),
            "avg_path": avg_path,
            "diameter": diameter,
            "assortativity_side": np.nan,  # not defined
            "assortativity_degree": nx.degree_assortativity_coefficient(Gs),
            "top5_degree": "",
            "top5_betweenness": "",
        }

    return [
        compute_stats(G_er, "ER"),
        compute_stats(G_ba, "BA")
    ]

In [ ]:
SNAPSHOTS = {
    "before": "G_full_co_thread_before.gexf",
    "during": "G_full_co_thread_during.gexf",
    "after":  "G_full_co_thread_after.gexf",
}

rows_real = []
for phase, path in SNAPSHOTS.items():
    print(f"\n=== {phase.upper()} (REAL) ===")
    G = nx.read_gexf(path)
    stats = summarize_graph(G, phase=phase)
    rows_real.append(stats)
    # stampa rapida
    print({k: stats[k] for k in ["nodes","edges","density","clustering","avg_path","diameter",
                                 "assortativity_side","assortativity_degree"]})
    print("Top5 Degree      →", stats["top5_degree"])
    print("Top5 Betweenness →", stats["top5_betweenness"])

df_real = pd.DataFrame(rows_real)
df_real

In [ ]:
# ==================================================
# BUILD FINAL REPORT (REAL vs SYNTHETIC MODELS)
# ==================================================

rows_models = []

# Generate ER and BA models for each phase
for phase in df_real["phase"].unique():

    # Extract number of nodes and edges from real graph
    n = int(df_real.loc[df_real["phase"] == phase, "nodes"].iloc[0])
    e = int(df_real.loc[df_real["phase"] == phase, "edges"].iloc[0])

    # Generate synthetic models (ER + BA)
    rows_models.extend(compare_models(n, e, phase))


# Convert models to DataFrame
df_models = pd.DataFrame(rows_models)


# ==================================================
# MERGE REAL DATA + MODELS
# ==================================================

# Combine real network metrics with synthetic benchmarks
df_report = pd.concat([df_real, df_models], ignore_index=True)


# ==================================================
# COLUMN ORDERING (for readability)
# ==================================================

preferred_columns = [
    "phase", "model", "nodes", "edges", "avg_degree", "density",
    "clustering", "avg_path", "diameter",
    "assortativity_side", "assortativity_degree",
    "top5_degree", "top5_betweenness"
]

# Keep preferred order + append any extra columns
ordered_columns = (
    [col for col in preferred_columns if col in df_report.columns] +
    [col for col in df_report.columns if col not in preferred_columns]
)

df_report = df_report[ordered_columns]


# ==================================================
# SAVE REPORT
# ==================================================

REPORT_CSV = "network_report.csv"

df_report.to_csv(REPORT_CSV, index=False)

print(f"✅ Report saved to: {REPORT_CSV}")